In [1]:
import pandas as pd
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
from gensim.models import FastText, Word2Vec
from itertools import product
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTHONWARNINGS'] = 'ignore'


nltk.download('punkt')
nltk.download('punkt_tab')


C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[nltk_data] Downloading package punkt to C:\Users\HP
[nltk_data]     Victus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\HP
[nltk_data]     Victus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [2]:
numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "benefits_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "benefits_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions"]

In [3]:
### loading clean text
df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")
df.head()


,full_text,telecommuting,missing_count,total_text_len,company_profile_len,description_len,requirements_len,benefits_len,company_profile_word_count,description_word_count,requirements_word_count,benefits_word_count,salary_provided,has_company_profile,vague_location,has_company_logo,has_questions,fraudulent
0,marketing intern us ny new york we are food 52...,0,3,2642,885,905,852,0,141,124,115,0,0,1,0,1,0,0
1,customer service cloud video production nz auc...,0,1,6088,1286,2077,1433,1292,153,315,200,227,0,1,0,1,0,0
2,commissioning machinery assistant cma us ia we...,0,6,2597,879,355,1363,0,141,50,164,0,0,1,0,1,0,0
3,account executive washington dc us dc washingt...,0,0,5425,614,2600,1429,782,85,346,176,97,0,1,0,1,0,0
4,bill review manager us fl fort worth spot sour...,0,0,3926,1628,1520,757,21,207,168,89,3,0,1,0,1,1,0


In [4]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [5]:
X_train_text = train_data['full_text']
X_train_numeric = train_data[numeric_cols]
y_train = train_data['fraudulent']

X_val_text = val_data['full_text']
X_val_numeric = val_data[numeric_cols]
y_val = val_data['fraudulent']

X_test_text = test_data['full_text']
X_test_numeric = test_data[numeric_cols]
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens

We will try a pretrained CBow as well as using FastText from scratch and evaluate their performance. 

#### FastText (using skipgram)

In [6]:
sentences = [word_tokenize(text.lower()) for text in X_train_text]  

In [ ]:
fasttext_model = FastText(vector_size=100, window=5, min_count=5, sg=1) #assuming we want a 100 word vector
fasttext_model.build_vocab(sentences)
fasttext_model.train(sentences, total_examples=fasttext_model.corpus_count, epochs=10)

Some checks if we we manage to learn technical jargon and correlated words

In [7]:
fasttext_model_optimal = FastText(vector_size=100, window=3, min_count=2, epochs=10, negative=5, sg = 1) 
fasttext_model_optimal.build_vocab(sentences)
fasttext_model_optimal.train(sentences, total_examples=fasttext_model_optimal.corpus_count, epochs=10)

(48263227, 63557210)

In [8]:
## Save the trained optimal FastText models
fasttext_model_optimal.save("optimal_fasttext.bin")

In [ ]:
print(fasttext_model.wv['saas']) #technical jargon
print(fasttext_model.wv.vectors.shape) 

In [ ]:
print(fasttext_model.wv['bingsu']) #checking for oov
print(fasttext_model.wv.vectors.shape) 

In [ ]:
print(fasttext_model.wv.similarity('software', 'engineer'))
print(fasttext_model.wv.similarity('skills', 'experience'))
print(fasttext_model.wv.most_similar('water', topn=10))

print(fasttext_model_optimal.wv.similarity('software', 'engineer'))
print(fasttext_model_optimal.wv.similarity('skills', 'experience'))
print(fasttext_model_optimal.wv.most_similar('water', topn=10))


#### CBOW

In [ ]:

cbow_model = Word2Vec(vector_size=100, window=5, min_count=5, sg=0) #assuming we want a 100 word vector
cbow_model.build_vocab(sentences)
cbow_model.train(sentences, total_examples=cbow_model.corpus_count, epochs=10)

In [ ]:
print(cbow_model.wv.similarity('software', 'engineer'))
print(cbow_model.wv.similarity('skills', 'experience'))
cbow_model.wv.most_similar('research', topn=10)


### Hyperparameter tuning

We are going to tune the `sliding window size`, `embedding vector size`,`epochs`, `negative sampling` to obtain the optimal custom embedding which would be easily plugged into our model

we will measure through extrinsic evaluation and intrinsic evaluation

In [ ]:
#to check oov rate
def oov_rate(model, corpus):
    oov = sum(1 for w in corpus if w not in model.wv)
    return oov/len(corpus)

#check top 10 words are similar to each other
def nearest_neighbour(model, test_words, topn = 10):
    scores = []
    for word in test_words:
        try:
            neighbours = model.wv.most_similar(word, topn=topn)
            scores.append(np.mean([score for _, score in neighbours]))
        except KeyError:
            pass
    return np.mean(scores)

#check if 2 correlated and 2 uncorrelated words are similar
def analogy_score(model, test_cases):
    correct = 0
    for pos1, pos2, neg1, expected in test_cases:
        try:
            results = model.wv.most_similar(
                positive=[pos1, pos2], negative=[neg1], topn=5
            )
            predicted = [w for w, _ in results]
            if expected in predicted:
                correct += 1
        except KeyError:
            pass
    return correct / len(test_cases)

In [ ]:
#DO NOT RUN THIS IT WILL TAKE MANY HOURS

param_grid = {
    'vector_size': [100, 200],
    'window':      [3, 5, 10],
    'min_count':   [2, 5],
    'epochs':      [10, 20],
    'negative':    [5, 10],
}

test_words = ['engineer', 'manager', 'python', 'healthcare', 'experience', 'salary']
job_analogies = [
    ('engineer', 'python', 'manager', 'java'),
    ('senior', 'engineer', 'junior', 'developer'),
    ('full_time', 'salary', 'part_time', 'hourly'),
    ('healthcare', 'nurse', 'finance', 'analyst'),
]
vocab = [word for sentence in sentences for word in sentence]

results = []

keys = list(param_grid.keys())
combos = list(product(*param_grid.values()))
print(f"Total combinations: {len(combos)} runs")

for combo in combos:
    params = dict(zip(keys, combo))
    model = FastText(**params, sg=1, min_n=3, max_n=6)


    model.build_vocab(sentences)
    model.train(sentences, total_examples=model.corpus_count, epochs=params['epochs'])

    coherence = nearest_neighbour(model, test_words)
    oov       = oov_rate(model, vocab)
    analogy   = analogy_score(model, job_analogies)

    results.append({
            **params,
            'coherence':  round(coherence, 4),
            'oov_rate':   round(oov, 4),
            'analogy':    round(analogy, 4),
        })
    print(f" {params} → coherence={coherence:.4f}, oov={oov:.4f}, analogy={analogy:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('data/clean/embedding_tuning.csv', index=False)